# Superstore Project 
Dataset containing Sales & Profits of a Superstore

-- Business Statistics & Insights | Master in Business Analytics & AI

The goal is to use the Superstore Sales dataset (9,994 transactions, 2014–2017) not just to complete the assignment, but to produce work that directly maps to the five skill areas in the target role. Each section below defines the analysis, the method, the business framing, and the free tool to use.

* Dataset: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

Metadata:
* Row ID => Unique ID for each row.
* Order ID => Unique Order ID for each Customer.
* Order Date => Order Date of the product.
* Ship Date => Shipping Date of the Product.
* Ship Mode=> Shipping Mode specified by the Customer.
* Customer ID => Unique ID to identify each Customer.
* Customer Name => Name of the Customer.
* Segment => The segment where the Customer belongs.
* Country => Country of residence of the Customer.
* City => City of residence of of the Customer.
* State => State of residence of the Customer.
* Postal Code => Postal Code of every Customer.
* Region => Region where the Customer belong.
* Product ID => Unique ID of the Product.
* Category => Category of the product ordered.
* Sub-Category => Sub-Category of the product ordered.
* Product Name => Name of the Product
* Sales => Sales of the Product.
* Quantity => Quantity of the Product.
* Discount => Discount provided.
* Profit => Profit/Loss incurred.

### Step 1. Data Preparation

1.1 — Load the data and do a first look

In [1]:
# Import libraries and dataset
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from IPython.display import display
import plotly.offline as py
import plotly.graph_objs as go
import plotly.tools as tls
import sqlalchemy as sa
from functools import reduce


warnings.filterwarnings("ignore")

# Import dataset from Kaggle 
import kagglehub
import shutil, os

"""cache_path = os.path.expanduser("~/.cache/kagglehub/datasets/vivek468/superstore-dataset-final")
shutil.rmtree(cache_path)
print("Cache cleared")"""

path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
df = pd.read_csv(f"{path}/Sample - Superstore.csv", encoding="latin-1")
print(df.head())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [5]:
# Make a copy of the original dataframe
superstore_raw = df.copy()

# Export to folder
os.makedirs('raw', exist_ok=True)
superstore_raw.to_csv('raw/df.csv', index=False)
          
print(f" Saved to dbt/raw/df.csv")

 Saved to dbt/raw/df.csv


#### Task 1.A - Dataset Understanding
* How many observations and variables are included?
* What types of variables exist?
    * Numerical
    * Categorical
    * Date variables

A.1.Observations & Variables

In [6]:
# Shape: how observations and variables?
print(f"Observations (rows): {df.shape[0]}")
print (f"Variables (columns): {df.shape[1]}")

Observations (rows): 9994
Variables (columns): 21


A.2. Variable Classification

In [7]:
# Column names and data types
numerical = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical = df.select_dtypes(include=['object']).columns.tolist()

print("NUMERICAL: ", numerical)
print("\nCATEGORICAL: ", categorical)
print("\nDATE variables to convert: , ['Order Date', 'Ship Date']")

NUMERICAL:  ['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']

CATEGORICAL:  ['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name']

DATE variables to convert: , ['Order Date', 'Ship Date']


A.3. Summary

In [8]:
variable_types = {
    'Variable': df.columns.tolist(),
    'Dtype': df.dtypes.astype(str).tolist(),
    'Category': []
}

date_vars = ['Order Date', 'Ship Date']
num_vars = ['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']

for col in df.columns:
    if col in date_vars:
        variable_types['Category'].append('Date')
    elif col in num_vars:
        variable_types['Category'].append('Numerical')
    else:
        variable_types['Category'].append('Categorical')
        
pd.DataFrame(variable_types)

,Variable,Dtype,Category
0,Row ID,int64,Numerical
1,Order ID,object,Categorical
2,Order Date,object,Date
3,Ship Date,object,Date
4,Ship Mode,object,Categorical
5,Customer ID,object,Categorical
6,Customer Name,object,Categorical
7,Segment,object,Categorical
8,Country,object,Categorical
9,City,object,Categorical


The dataset contains 9994 observations and 21 variables. Variables are classified into three types: 6 numerical (Sales, Profit, Quantity, Discount, Row ID, Postal Code), 2 Dates variables (Order Date, Ship Date - stores as object strings and requiring convertion), and 13 categorical variables including Customer Segment, Product Category, Region, and Ship Mode. The Postal Code was stored as number, is a nominal identifier and should not be used in arithmetic operations.

#### 1.B. Data Quality Assessment

B1. - Check for missing values

In [9]:
missing = df.isnull().sum()
missing_pct = (missing /len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

if missing_report.empty:
    print("No missing values found in any column.")
else:
    print(missing_report)

No missing values found in any column.


No missing values were detected in any of the 21 variables. This eliminates the need for imputation and confirms the dataset is complete for statistical analysis.

B2. - Duplicates records

In [10]:
# Full rows duplicates (data errors)
full_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {full_dupes}")

# Duplicate Order IDs (one order = multiple line items)
order_dupes = df['Order ID'].duplicated().sum()
unique_orders = df['Order ID'].nunique()
print(f"Unique Order IDs: {unique_orders}")
print(f"Rows sharing an Order ID: {order_dupes}")
print(f"Average line items per order: {len(df) / unique_orders:.1f}")

Exact duplicate rows: 0
Unique Order IDs: 5009
Rows sharing an Order ID: 4985
Average line items per order: 2.0


No exact duplicate rows were found. The dataset intentionally contains repeated Order IDs: each order can contain multiple products, resulting in multiple rows per order. The 9,994 rows correspond to 5,009 unique orders, averaging approximately 2 line items per order. This structure must be accounted for when aggregating at the order level — summing by Order ID rather than counting rows.

B3. Incosistent categories

In [11]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

B4. Data type issues

B5. Outliers

#### 1.C. Variable Transformation

### Step 2. Descriptive Statistics